# EDA 01 — Raw metadata files exploration

This notebook performs an initial exploratory analysis of the raw metadata files used in the project.

The objective is to inspect the structure, dimensions, columns, missing values and basic content of the three original CSV files stored in `data/raw/metadata`.

No preprocessing or feature engineering is performed in this notebook. The goal is only to understand the raw input data before defining the preprocessing pipeline.

In [9]:
# Libraries
from skin_lesion_ai.utils.config import load_raw_metadata
import pandas as pd

In [3]:
# Loading data
df1, df2, df3 = load_raw_metadata()
print("df1 - ground_truth:", df1.shape)
print("df2 - supplement:", df2.shape)
print("df3 - metadata:", df3.shape)

/Users/carlesraichbros/my-image-classifier/src/skin_lesion_ai/utils/config.py:40: DtypeWarning: Columns (0: iddx_5) have mixed types. Specify dtype option on import or set low_memory=False.
  supplement = pd.read_csv(path(raw["supplement_csv"]))


df1 - ground_truth: (401059, 2)
df2 - supplement: (401059, 13)
df3 - metadata: (401059, 42)


## ground_truth (df1)

In [ ]:
# Data Wrangler
df1

In [ ]:
# malignant col
df1["malignant"].unique().tolist()

[0.0, 1.0]

In [11]:
# malignant proportions
malignant_summary = (
    df1["malignant"]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / x["count"].sum(), 2))
)

malignant_summary.loc["Total"] = [
    malignant_summary["count"].sum(),
    malignant_summary["percentage"].sum(),
]

malignant_summary

,count,percentage
malignant,,
0.0,400666.0,99.9
1.0,393.0,0.1
Total,401059.0,100.0


### ground_truth (df1) summary

Basic lesion labeling. 

- **Rows**: 401059. No duplicated rows.
- **Cols**: 2
- **Col names**: [isic_id, malignant]

    - *isic_id*: unique id per lesion (pkey). 0 NA.  

    - *malignant*: malignancy flag per lesion. 0 NA. Two values: 
        - 0 = not malignant (400,666 counts; 99.9%). 
        - 1 = malignant (393 counts; 0.1%).

## supplement (df2)

In [ ]:
# Data Wrangler
df2

In [8]:
df2.columns.tolist()

['isic_id',
 'attribution',
 'copyright_license',
 'lesion_id',
 'iddx_full',
 'iddx_1',
 'iddx_2',
 'iddx_3',
 'iddx_4',
 'iddx_5',
 'mel_mitotic_index',
 'mel_thick_mm',
 'tbp_lv_dnn_lesion_confidence']

In [4]:
# isic_id col
isic_id_df1 = set(df1["isic_id"])
isic_id_df2 = set(df2["isic_id"])

print("Only in df1:", isic_id_df1 - isic_id_df2)
print("Only in df2:", isic_id_df2 - isic_id_df1)

Only in df1: set()
Only in df2: set()


In [6]:
# attribution col
df2["attribution"].unique().tolist()

['Memorial Sloan Kettering Cancer Center',
 'ACEMID MIA',
 'Department of Dermatology, Hospital Clínic de Barcelona',
 'University Hospital of Basel',
 'Frazer Institute, The University of Queensland, Dermatology Research Centre',
 'Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris',
 'ViDIR Group, Department of Dermatology, Medical University of Vienna']

In [12]:
# attribution proportions
attribution_summary = (
    df2["attribution"]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / x["count"].sum(), 2))
)

attribution_summary.loc["Total"] = [
    attribution_summary["count"].sum(),
    attribution_summary["percentage"].sum(),
]

attribution_summary

,count,percentage
attribution,,
Memorial Sloan Kettering Cancer Center,129068.0,32.18
"Department of Dermatology, Hospital Clínic de Barcelona",105724.0,26.36
University Hospital of Basel,65218.0,16.26
"Frazer Institute, The University of Queensland, Dermatology Research Centre",51768.0,12.91
ACEMID MIA,28665.0,7.15
"ViDIR Group, Department of Dermatology, Medical University of Vienna",12640.0,3.15
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",7976.0,1.99
Total,401059.0,100.00


In [8]:
# copyright_licese col
df2["copyright_license"].unique().tolist()

['CC-BY', 'CC-0', 'CC-BY-NC']

In [10]:
# license per site
pd.crosstab(df2["attribution"], df2["copyright_license"])

copyright_license,CC-0,CC-BY,CC-BY-NC
attribution,,,
ACEMID MIA,28665,0,0
"Department of Dermatology, Hospital Clínic de Barcelona",0,0,105724
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",0,7976,0
"Frazer Institute, The University of Queensland, Dermatology Research Centre",0,51768,0
Memorial Sloan Kettering Cancer Center,0,129068,0
University Hospital of Basel,0,0,65218
"ViDIR Group, Department of Dermatology, Medical University of Vienna",0,0,12640


In [16]:
# lesion_id col

n_null = df2["lesion_id"].isna().sum()
n_non_null = df2["lesion_id"].notna().sum()
n_dup = df2.loc[df2["lesion_id"].notna(), "lesion_id"].duplicated().sum()

lesion_id_summary = pd.DataFrame(
    {
        "count": [n_null, n_non_null, len(df2), n_dup],
        "percentage": [
            round(100 * n_null / len(df2), 2),
            round(100 * n_non_null / len(df2), 2),
            100.0,
            round(100 * n_dup / n_non_null, 2),
        ],
    },
    index=["Null", "Non-null", "Total", "Non-null duplicated"],
)

lesion_id_summary

,count,percentage
Null,379001,94.5
Non-null,22058,5.5
Total,401059,100.0
Non-null duplicated,0,0.0


### supplement (df2) summary

Detailed lesion labeling.

- **Rows**: 401059. No duplicated rows.
- **Cols**: 13.
- **Col names**: ['isic_id','attribution','copyright_license','lesion_id','iddx_full','iddx_1''iddx_2','iddx_3','iddx_4','iddx_5','mel_mitotic_index','mel_thick_mm', 'tbp_lv_dnn_lesion_confidence'].

   - *isic_id*: unique id per lesion (pkey). 0 NA. 100% match with df1.

   - *attribution*: lesion site of origin. 0 NA. 7 sites (DESC order):

      - Memorial Sloan Kettering Cancer Center (USA): 129,086 lesions (32.18%).
      - Department of Dermatology, Hospital Clínic de Barcelona (Spain): 105,274 lesions (26.36%).
      - University Hospital of Basel (Switzerland): 65,218 lesions (16.26%).
      - Frazer Institute, The University of Queensland, Dermatology Research Centre (Australia): 65,218 lesions (12.91%).
      - ACEMID MIA (Australian Centre of Excellence in Melanoma Imaging and Diagnosis - Melanoma Institute Australia), which includes Alfred Hospital and FNQH Cairn (Australia): 28,665 lesions (7.15%).
      - ViDIR Group, Department of Dermatology, Medical University of Vienna (Austria): 12,640 lesions (3.15%).
      - Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris (Greece): 7,976 lesions (1.99%).

   - *copyright_license*: 3 types of copyright license. 0 NA.

      - *CC-0 (Creative Commons Zero)*: Public domain dedication. The data can be used, modified, and redistributed without attribution and without restrictions. This applies to:
         - ACEMID MIA

      - *CC-BY (Creative Commons Attribution)*: The data can be used, modified, and redistributed, including for commercial purposes, provided that appropriate attribution is given to the original source. This applies to:
         - Memorial Sloan Kettering Cancer Center
         - Frazer Institute, The University of Queensland, Dermatology Research Centre
         - Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris

      - *CC-BY-NC (Creative Commons Attribution–NonCommercial)*: The data can be used, modified, and redistributed with attribution, but commercial use is not permitted. This applies to:
         - Department of Dermatology, Hospital Clínic de Barcelona
         - University Hospital of Basel
         - ViDIR Group, Department of Dermatology, Medical University of Vienna


## metadata (df3)